# Preliminaries

In [ ]:
import os
from pathlib import Path
import re
import warnings

import contextily as cx
import fiona
import geobr
import numpy as np
import osmnx as ox
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from rasterstats import zonal_stats
import richdem as rd
import rioxarray
from scipy.stats.mstats import hmean

%matplotlib inline
%config InlineBackend.figure_format='retina'

In [ ]:
DATABASE = os.environ.get('DB_FOLDER')
DATABASE = Path(DATABASE)

OUT_FOLDER = os.environ.get('OUT_FOLDER')
OUT_DIR = Path(OUT_FOLDER) / 'A'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Declividade

A declividade é calculada a partir de um [Modelo Digital de Terreno do Estado de Minas Gerais](https://idesisema.meioambiente.mg.gov.br/geonetwork/srv/api/records/e2544c02-65f3-4380-b78e-76606f08653b). A partir dele, será recortado o município de Belo Horizonte e será obtida a declividade média do terreno, a ser então imputada nas células hexagonais. 

In [ ]:
# Limites municipais para extrair BH
# do raster de Minas Gerais
bh = geobr.read_municipality(3106200)

In [ ]:
# Lendo raster (EPSG: 4674)
inpath = DATABASE / 'beaga/dem_minas_gerais.tiff'
dem_mg = rioxarray.open_rasterio(inpath)

In [ ]:
dem_bh = dem_mg.rio.clip(
    bh.geometry.to_numpy(),
    bh.crs,
    )

del dem_mg

Reprojetar é importante para evitar os problemas no cálculo da declividade descritos [aqui](https://stackoverflow.com/questions/70835337/python-calculate-slope-raster-from-dem). Técnicas para fazer isso são apresentadas [aqui](https://www.earthdatascience.org/courses/use-data-open-source-python/intro-raster-data-python/raster-data-processing/reproject-raster/).

In [ ]:
# Reprojetando e salvando o raster
dem_bh = dem_bh.rio.reproject(bh.to_crs(31983).crs)

outpath = OUT_DIR / 'dem_bh.tiff'
dem_bh.rio.to_raster(outpath)

In [ ]:
slope = rd.TerrainAttribute(
    rd.LoadGDAL(outpath, no_data=-33),
    attrib='slope_percentage',
    )

In [ ]:
rd.SaveGDAL(outpath, slope)

In [ ]:
inpath = OUT_DIR / 'oportunidades/rais/establishments_by_hex.parquet'
hexes = gpd.read_parquet(inpath).query("ano == 2023").drop(columns='ano')

hexes = hexes.reset_index(drop=True)

In [ ]:
slopes = []
for r, h in hexes.groupby('aperture'):
    slope_by_hex = zonal_stats(
        h,
        outpath,  # Path to surface file
        all_touched=True,
        stats=['mean', 'max', 'min', 'std', 'median'],
    )
    slope_by_hex = pd.DataFrame(slope_by_hex).assign(aperture=r)
    slopes.append(
        slope_by_hex.set_index(h.hex_id)
        )

del slope_by_hex

slopes = pd.concat(slopes)

In [ ]:
slopes.sample(5)

In [ ]:
slope_by_hex = (
    hexes
    .reindex(columns=['hex_id', 'geometry'])
    .merge(
        slopes,
        left_on='hex_id',
        right_index=True,
        )
    )

In [ ]:
slope_by_hex.head()

In [ ]:
slope_by_hex.query("aperture == 11").explore('mean', cmap='gist_earth', prefer_canvas=True, )

In [ ]:
outpath = OUT_DIR / 'slopes_by_hex.parquet'
slope_by_hex.to_parquet(outpath)